Landing table + simulated insert

In [0]:
%sql
CREATE TABLE IF NOT EXISTS sensor_raw_landing (
  device_id STRING, event_time TIMESTAMP, temperature DOUBLE, humidity DOUBLE
);

INSERT INTO sensor_raw_landing
SELECT
  concat('device_', cast(int(rand()*10) as string)) AS device_id,
  current_timestamp() AS event_time,
  20 + rand()*15 AS temperature,
  30 + rand()*40 AS humidity
FROM range(50);

In [0]:
%sql
SELECT * FROM sensor_raw_landing ORDER BY event_time DESC LIMIT 10;

In [0]:
%sql
INSERT INTO sensor_raw_landing
SELECT
  concat('device_', cast(int(rand()*10) as string)) AS device_id,
  current_timestamp() AS event_time,
  20 + rand()*15 AS temperature,
  30 + rand()*40 AS humidity
FROM range(50);

In [0]:
%sql
SELECT count(*) FROM sensor_raw_landing;

Bronze

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE bronze_sensor_readings
TRIGGER ON UPDATE AT MOST EVERY INTERVAL 1 minute
AS SELECT * FROM STREAM sensor_raw_landing;

In [0]:
%sql
SELECT count(*) FROM bronze_sensor_readings;

Silver

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE silver_sensor_readings
TRIGGER ON UPDATE AT MOST EVERY INTERVAL 1 minute
AS SELECT *
FROM STREAM bronze_sensor_readings WATERMARK event_time DELAY OF INTERVAL 2 MINUTES
WHERE temperature BETWEEN 0 AND 50;

In [0]:
%sql
SELECT count(*) FROM silver_sensor_readings;

Gold

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE gold_sensor_5min_avg
TRIGGER ON UPDATE AT MOST EVERY INTERVAL 1 minute
AS SELECT
  window(event_time, '5 minutes') AS time_window,
  device_id,
  avg(temperature) AS avg_temp,
  avg(humidity) AS avg_humidity
FROM STREAM bronze_sensor_readings WATERMARK event_time DELAY OF INTERVAL 2 MINUTES
WHERE temperature BETWEEN 0 AND 50
GROUP BY window(event_time, '5 minutes'), device_id;

Can't run 3 pipelines at once due to resource cap, so we drop the bronze table and only keep the silver and gold tables

In [0]:
%sql
-- Free up a pipeline slot
DROP TABLE  IF EXISTS bronze_sensor_readings;
DROP TABLE IF EXISTS silver_sensor_readings;

-- Redefine silver to read straight from the landing table
CREATE OR REFRESH STREAMING TABLE silver_sensor_readings
TRIGGER ON UPDATE AT MOST EVERY INTERVAL 1 minute
AS SELECT *
FROM STREAM sensor_raw_landing WATERMARK event_time DELAY OF INTERVAL 2 MINUTES
WHERE temperature BETWEEN 0 AND 50;

In [0]:
%sql
SELECT count(*) FROM silver_sensor_readings;

Then the aggr gold

In [0]:
%sql
DROP TABLE gold_sensor_5min_avg;

CREATE OR REFRESH STREAMING TABLE gold_sensor_5min_avg
TRIGGER ON UPDATE AT MOST EVERY INTERVAL 1 minute
AS SELECT
  window(event_time, '5 minutes') AS time_window,
  device_id,
  avg(temperature) AS avg_temp,
  avg(humidity) AS avg_humidity
FROM STREAM silver_sensor_readings WATERMARK event_time DELAY OF INTERVAL 2 MINUTES
GROUP BY window(event_time, '5 minutes'), device_id;

In [0]:
%sql
SELECT count(*) FROM gold_sensor_5min_avg;

In [0]:
%sql
SELECT * FROM gold_sensor_5min_avg ORDER BY time_window, device_id;

In [0]:
%sql
SELECT max(event_time) AS latest_event, current_timestamp() AS now_time
FROM silver_sensor_readings;

In [0]:
%sql
SELECT min(event_time), max(event_time), count(distinct device_id) FROM silver_sensor_readings;

In [0]:
%sql
DELETE FROM sensor_raw_landing;

In [0]:
%sql
SELECT * FROM gold_sensor_5min_avg ORDER BY time_window, device_id;